# 06 · Narrator and numeric grounding

**Role of this layer.** The narrator writes a short plain-English summary. It
never sees data series, only a **facts sheet** of numbers the executor computed,
already rounded (4 significant figures). It is told to use only those numbers,
exactly as displayed, and to do no arithmetic.

`check_grounding` then verifies that deterministically: every number in the
text must be a correct rounding of a computed fact (or its percentage form), a
plan parameter, a number from the request or a warning, an annualisation
factor, or a date within the data range; the 5% significance and 95% confidence
levels count only when a significance test ran. Without an API key a
deterministic template writes the narrative instead, and it passes the same check.

That check is guardrail **N01 UNGROUNDED_NARRATIVE** (action WARN, enforced by the
narrator check): an LLM narrative that fails it is withheld (kept in the run
record as `narrative.withheld`), the template narrative is shown instead, and an
N01 issue lists the numbers that failed. The numerical results are not affected.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.narrator import check_grounding

scenario = SCENARIOS_BY_ID["volatility_flag_success"]  # high-volatility weeks, then a test
session = run_request(scenario.request, list(scenario.dataset_ids), "mock", approve=True)
facts = session.narrative.facts
pd.DataFrame([{"fact": f.fact_id, "value": f.value, "display": f.display} for f in facts]).head(20)

,fact,value,display
0,s1.n_obs,14042,"14,042"
1,s1.start,1971-02-05,1971-02-05
2,s1.end,2026-10-15,2026-10-15
3,s1.last_value,26713.03,"26,713"
4,s2.n_obs,2906,"2,906"
5,s2.start,1971-02-05,1971-02-05
6,s2.end,2026-10-09,2026-10-09
7,s2.last_value,27366.17,"27,366"
8,s3.n_obs,2905,"2,905"
9,s3.start,1971-02-12,1971-02-12


In [3]:
narrative = session.narrative
print(f"source: {narrative.source}")
print(narrative.text)
print("\ngrounding:", narrative.grounding)

source: template
The analysis used NASDAQCOM (1971-02-05 to 2026-10-15). The latest rolling volatility (s4) is 0.1368 on 2026-10-09. 446 of 2,842 observations (a share of 0.1569) were flagged, in 40 distinct episodes. Comparing flagged with normal periods (s7), the mean is 0.002093 versus 0.002302 (difference -0.0002085; standard deviation 0.04482 versus 0.02316), over 446 flagged observations in 40 distinct episodes and 2,395 normal observations; this is descriptive, not a statistical test. The significance test (s8) finds no statistically significant difference at the 5% level in 1-period forward returns between flagged and normal periods: the difference in means is -0.0002085 (normal periods 0.002302), with a Newey-West standard error of 0.002229 (8 lags), a p-value of 0.9255 and a 95% confidence interval from -0.004578 to 0.004161. The 446 flagged observations form 40 distinct episodes, the real measure of independent evidence because flagged periods cluster; the test describes an 

The check catches invented numbers, and accepts correct roundings and
percentages. (`s4.last_value` is the latest volatility in the facts sheet.)

In [4]:
vol = next(f for f in facts if f.fact_id == "s4.last_value").display
share = next(f for f in facts if f.fact_id == "s5.share_flagged").display
p_value = next(f for f in facts if f.fact_id == "s8.p_value").display
for sentence in [
    f"Latest volatility is {vol}, and a share of {share} of the weeks was flagged.",
    f"Latest volatility is about {float(vol) * 100:.1f}%.",
    f"No statistically significant difference at the 5% level (p-value {p_value}).",
    f"The p-value is {float(p_value) - 0.1:.4f}.",  # a wrong value
    "The Sharpe ratio of the strategy is 1.83.",
    "Volatility peaked in 1955.",
]:
    result = check_grounding(sentence, facts, session.plan, [scenario.request])
    print(f"{'OK ' if result.grounded else 'BAD'} {sentence}  ungrounded={result.ungrounded}")

OK  Latest volatility is 0.1368, and a share of 0.1569 of the weeks was flagged.  ungrounded=[]
OK  Latest volatility is about 13.7%.  ungrounded=[]
OK  No statistically significant difference at the 5% level (p-value 0.9255).  ungrounded=[]
BAD The p-value is 0.8255.  ungrounded=['0.8255']
BAD The Sharpe ratio of the strategy is 1.83.  ungrounded=['1.83']
BAD Volatility peaked in 1955.  ungrounded=['1955']


With an API key, the same facts go to the LLM narrator (live mode):

In [5]:
if settings.has_api_key:
    from quantlab.narrator import narrate

    live = narrate(
        session.request,
        session.plan,
        session.execution.artifacts,
        session.warnings(),
        "live",
        settings,
    )
    print(live.source, live.model, live.error)
    print(live.text)
    print("grounding:", live.grounding)
    print("N01 issues:", [issue.message for issue in live.issues])
else:
    print(
        f"No {settings.api_key_name} set: skipping the LLM narrator (the template is used above)."
    )

No OPENAI_API_KEY set: skipping the LLM narrator (the template is used above).
